# 集合覆盖 scp41（最小成本）

**问题**：实例来自 OR-Library 的 scp41：m=200 个行元素，n=1000 个列集合。列 j 的成本为 c_j，覆盖的行集合为 S_j（a_ij=1 表示列 j 覆盖行 i）。目标是选择一组列，使每一行至少被一个选中列覆盖，同时总成本最小。

**数学模型**

$$\min \sum_{j=1}^{n} c_j x_j$$

$$\text{s.t.}\quad \sum_{j: i \in S_j} x_j \ge 1,\quad i=1,\dots,m$$

$$x_j\in\{0,1\},\quad j=1,\dots,n$$

数据文件：\`/mnt/d/exactTest/column-generation-testcases/set_covering/scp41.txt\`。文献最优值 429（本套件用直接 MIP 自证）。

## 方法：列生成（池扫描定价）

**主问题（受限主问题 RMP，LP 松弛）**

$$\min \sum_{j\in P} c_j x_j + M\sum_{i=1}^{m} a_i$$

$$\text{s.t.}\quad a_i + \sum_{j\in P: i\in S_j} x_j \ge 1,\quad i=1,\dots,m$$

$$x_j\ge 0,\quad a_i\ge 0$$

其中 a_i 是人工列（成本 M=Σc+1），保证 RMP 初始可行；P 是已生成列池。

**定价子问题（池扫描）**

$$\bar c_j = c_j - \sum_{i\in S_j}\pi_i$$

扫描全部 1000 个候选列，取 reduced cost 最小的列；若 $\min_j \bar c_j < -10^{-7}$ 则加入 RMP，否则 LP 收敛。

**原理要点**

1. RMP 用 MathOpt + GLOP 求 LP 最优，读取行约束对偶 $\pi$。
2. 池完整（P=全部 1000 列）时，逐列扫描定价与动态定价等价：动态定价的可行域就是这 1000 列，扫描取最负者即最优定价列。
3. 每次把最负 reduced cost 列加入 RMP，迭代至无负 reduced cost 列。
4. 收敛后得到完整 LP 松弛最优值；再用 HIGHS 在完整池上解整数 MIP 修复。
5. 停机：定价容差 1e-7、迭代上限 2000、总墙钟 110s。

In [1]:
import platform, time, datetime, math, ortools
from ortools.math_opt.python import mathopt

print("python", platform.python_version(), "| ortools", ortools.__version__)

DATA = "/mnt/d/exactTest/column-generation-testcases/set_covering/scp41.txt"
toks = open(DATA).read().split()
m, n = map(int, toks[:2])
costs = list(map(int, toks[2:2+n]))
idx = 2 + n
rows = []
for _ in range(m):
    k = int(toks[idx]); idx += 1
    rows.append([int(t)-1 for t in toks[idx:idx+k]]); idx += k
assert idx == len(toks)
colrows = [[] for _ in range(n)]
for i, row in enumerate(rows):
    for j in row:
        colrows[j].append(i)
print("m,n =", m, n, "| rows parsed =", len(rows), "| tokens consumed =", idx)


python 3.10.20 | ortools 9.15.6755
m,n = 200 1000 | rows parsed = 200 | tokens consumed = 5211


In [2]:
M = sum(costs) + 1
model = mathopt.Model(name="scp41_rmp")
a = [model.add_variable(lb=0.0, ub=float('inf'), is_integer=False, name=f"a{i}") for i in range(m)]
row_cons = [model.add_linear_constraint(a[i] >= 1.0, name=f"cov{i}") for i in range(m)]
xvar = [None]*n
added = set()
obj_terms = [(a[i], float(M)) for i in range(m)]
model.minimize_linear_objective(sum(coef*var for var, coef in obj_terms))
lp_params = mathopt.SolveParameters(enable_output=False)
tol = 1e-7
max_iter = 2000
t0 = time.perf_counter()
iters = 0
lp_obj = None
min_rc = float('inf')
hist = []
for it in range(max_iter):
    res = mathopt.solve(model, mathopt.SolverType.GLOP, params=lp_params)
    lp_obj = res.objective_value()
    pi = res.dual_values(row_cons)
    if not isinstance(pi, list):
        pi = [pi[c] for c in row_cons]
    min_rc = float('inf'); best = -1
    for j in range(n):
        if j in added:
            continue
        rc = costs[j]
        for i in colrows[j]:
            rc -= pi[i]
        if rc < min_rc:
            min_rc = rc; best = j
    hist.append((it+1, lp_obj, min_rc, best, len(added)))
    if min_rc >= -tol:
        iters = it+1
        break
    v = model.add_variable(lb=0.0, ub=float('inf'), is_integer=False, name=f"x{best}")
    xvar[best] = v
    for i in colrows[best]:
        row_cons[i].set_coefficient(v, 1.0)
    added.add(best)
    obj_terms.append((v, float(costs[best])))
    model.minimize_linear_objective(sum(coef*var for var, coef in obj_terms))
    iters = it+1
    if time.perf_counter()-t0 > 110:
        print("CG time limit reached at iteration", iters)
        break
cg_wall = time.perf_counter()-t0
res = mathopt.solve(model, mathopt.SolverType.GLOP, params=lp_params)
lp_obj = res.objective_value()
a_vals = res.variable_values(a)
print("CG iterations:", iters, "| columns added:", len(added))
print("RMP LP objective:", lp_obj)
print("last min reduced cost:", min_rc, "| cg_wall:", round(cg_wall, 3))
print("artificial positive count:", sum(v > 1e-7 for v in a_vals), "| max artificial:", max(a_vals))
print("first history (iter, lp_obj, min_rc, col, pool_size):", hist[:3])
print("last history:", hist[-3:])

# integer repair on the complete pool
t1 = time.perf_counter()
mip = mathopt.Model(name="scp41_cg_repair")
x = [mip.add_variable(lb=0.0, ub=1.0, is_integer=True, name=f"x{j}") for j in range(n)]
mip.minimize_linear_objective(sum(costs[j]*x[j] for j in range(n)))
for i, row in enumerate(rows):
    mip.add_linear_constraint(sum(x[j] for j in row) >= 1.0, name=f"cov{i}")
mres = mathopt.solve(mip, mathopt.SolverType.HIGHS, params=mathopt.SolveParameters(time_limit=datetime.timedelta(seconds=120), enable_output=False))
mv = mres.variable_values(x)
sel = [j for j in range(n) if mv[j] > 0.5]
print("integer repair:", mres.termination.reason, "| obj:", mres.objective_value(), "| best_bound:", mres.best_objective_bound(), "| wall:", round(time.perf_counter()-t1, 3))
print("selected_columns:", sorted(sel))
gap = (mres.objective_value() - lp_obj) / mres.objective_value() if mres.objective_value() else 0.0
print("LP-integer gap:", gap)


CG iterations: 166 | columns added: 165
RMP LP objective: 429.0
last min reduced cost: 0.0 | cg_wall: 2.905
artificial positive count: 0 | max artificial: 0.0
first history (iter, lp_obj, min_rc, col, pool_size): [(1, 10010200.0, -550549.0, 121, 0), (2, 9459651.0, -500433.0, 767, 1), (3, 8959218.0, -450442.0, 179, 2)]
last history: [(164, 429.0, -2.0, 141, 163), (165, 429.0, -1.0, 52, 164), (166, 429.0, 0.0, 23, 165)]
integer repair: TerminationReason.OPTIMAL | obj: 429.0 | best_bound: 429.0 | wall: 0.112
selected_columns: [0, 1, 2, 4, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19, 20, 21, 22, 24, 25, 27, 28, 42, 43, 45, 46, 47, 48, 49, 51, 53, 57, 58, 61, 62, 65, 68, 69, 70, 74, 76, 77, 80, 84, 85, 88, 90, 93, 102, 106, 115, 119, 120, 121, 123, 128, 137, 142, 143, 145, 152, 193, 274, 432]
LP-integer gap: 0.0


## 运行结果与结论

上方输出显示：CG 在 166 次定价迭代后收敛，RMP LP 最优值 **429.0**；人工列全部为 0；完整池上整数修复 HIGHS 证明整数最优 **429.0**。由于 LP 最优值 = 整数最优值，LP-integer gap = 0。

**基准最优值来源**：直接 MIP（01_direct）证明最优值 429.0；本方法 LP 下界与整数修复上界均为 429.0，也证明最优。

## 结论

集合覆盖是列生成的天然主场：覆盖约束的行对偶直接给出列定价公式。本实例池完整，池扫描定价与动态定价等价；LP 松弛恰好整数，CG 下界即为最优值。